## Observações

- As colunas loci representam genótipos e estão codificados categoricamente como `{0, 1, 2}`.
    - `O` para homozigoto de referência, `1` para heterozigoto e `2` para homozigoto alternativo.
- 97,5% dos loci apresentam os três genótipos possíveis, enquanto 2,5% não apresentam um dos genótipos na amostra.
- O alvo `Long_COVID` apresenta desbalanceamento moderado, com 33% de casos negativos e 67% de casos positivos.
- Dataset composto predominantemente por variantes comuns, mas com boa representatividade dos alelos minoritários:
    - Concentração moderada na categoria dominante, com mediana de 72% entre os loci.
    - MAF (frequência do alelo minoritário) mediana de 16% e mínimo de 4%, sugerindo quantidade razoável de alelos minoritários.
- As estatísticas de diversidade são saudáveis entre cromossomos, indicando que não há concentração de loci com baixa diversidade.
- O cromossomo X aparenta ser o mais diverso, com menor taxa de categoria dominante e maior MAF médio.

## Decisões

- Manter todos os loci
- Usar divisão estratificada para validação cruzada.
- Priorizar classes minoritárias (e.g., `class_weight="balanced"`) em modelos compatíveis.
- Avaliar modelos com métricas robustas a desbalanceamento:
    - **balanced accuracy** (foi a mais robusta nos meus testes)
    - F1
    - ROC AUC
    - PR AUC
    - sensibilidade
    - especificidade
    - matriz de confusão

In [ ]:
import pandas as pd

from covid.constants import LONG_COVID_RAW_DATA_PATH
from covid.eda.univariate import imbalance_summary, chromosomes_imbalance_summary

df = pd.read_csv(LONG_COVID_RAW_DATA_PATH)
df.shape

In [ ]:
df.describe().T.round(2)

In [ ]:
df.nunique().sort_values(ascending=False)

In [ ]:
from covid import feature

loci_columns = feature.get_loci_columns(df)

df[loci_columns].nunique().plot(
    kind="hist",
    title="Histogram of number of unique values in each column",
    xlabel="Number of Unique Values",
    grid=True,
)

In [ ]:
df[loci_columns].nunique().eq(3).sum() / len(loci_columns)

In [ ]:
cols = ["chr1_962184", "chr1_953259", "chr1_32695611"]
for col in cols:
    print(df[col].value_counts().to_string())

In [ ]:
import seaborn as sns

sns.set_style("whitegrid")

ax = sns.countplot(data=df, x=feature.TARGET, hue=feature.TARGET)
for container in ax.containers:
    ax.bar_label(container)

## Imbalance Analysis

In [ ]:
imbalance_summary = imbalance_summary(df=df, categorical_columns=loci_columns, rare_threshold=0.03)
imbalance_summary

In [ ]:
imbalance_summary.describe(percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]).T.round(2)

In [ ]:
imbalance_summary.hist(bins=30, figsize=(12, 8), grid=True)

In [ ]:
chromosomes_imbalance_summary(df=df, loci_columns=loci_columns)